In [3]:
!pip install rdkit

In [4]:
import pandas as pd
df = pd.read_csv('/content/S1PR1_dataset.csv')
df.head()

,Molecule ChEMBL ID,Smiles,Standard Value,Standard Type,pChEMBL Value,Activity_Class
0,CHEMBL4092538,CCc1ccc(COc2ccc3c(c2)CCC(CN2CC(C(=O)O)C2)=C3C)...,0.58,EC50,9.24,Active
1,CHEMBL389033,CCCCCCCCc1cccc(NC(=O)[C@H](N)CCP(=O)(O)O)c1,30.00,Ki,7.52,Active
2,CHEMBL5884223,N=C(NC(=O)C(NC(=O)c1ccc(Cl)c(Cl)c1)c1ccc(Cl)c(...,426.00,IC50,6.37,Active
3,CHEMBL5758543,CC(C)(C)OC(=O)NC(C(=O)NC(C(=O)NC(=N)c1ccccc1)c...,475.00,IC50,6.32,Active
4,CHEMBL6053560,CCOC(=O)NC(C(=O)NC(C(=O)NC(=N)c1ccccc1)c1cccc(...,1200.00,IC50,5.92,Inactive


In [5]:
# Keep only SMILES and Activity_Class
data = df[['Smiles', 'Activity_Class']].copy()
data.head()

,Smiles,Activity_Class
0,CCc1ccc(COc2ccc3c(c2)CCC(CN2CC(C(=O)O)C2)=C3C)...,Active
1,CCCCCCCCc1cccc(NC(=O)[C@H](N)CCP(=O)(O)O)c1,Active
2,N=C(NC(=O)C(NC(=O)c1ccc(Cl)c(Cl)c1)c1ccc(Cl)c(...,Active
3,CC(C)(C)OC(=O)NC(C(=O)NC(C(=O)NC(=N)c1ccccc1)c...,Active
4,CCOC(=O)NC(C(=O)NC(C(=O)NC(=N)c1ccccc1)c1cccc(...,Inactive


In [6]:
# Map Activity_Class to numeric labels
data['Activity'] = data['Activity_Class'].map({'Active': 1, 'Inactive': 0})
data.head()

,Smiles,Activity_Class,Activity
0,CCc1ccc(COc2ccc3c(c2)CCC(CN2CC(C(=O)O)C2)=C3C)...,Active,1
1,CCCCCCCCc1cccc(NC(=O)[C@H](N)CCP(=O)(O)O)c1,Active,1
2,N=C(NC(=O)C(NC(=O)c1ccc(Cl)c(Cl)c1)c1ccc(Cl)c(...,Active,1
3,CC(C)(C)OC(=O)NC(C(=O)NC(C(=O)NC(=N)c1ccccc1)c...,Active,1
4,CCOC(=O)NC(C(=O)NC(C(=O)NC(=N)c1ccccc1)c1cccc(...,Inactive,0


In [7]:
# Drop the original Activity_Class column
data = data.drop(columns=['Activity_Class'])
data.head()

,Smiles,Activity
0,CCc1ccc(COc2ccc3c(c2)CCC(CN2CC(C(=O)O)C2)=C3C)...,1
1,CCCCCCCCc1cccc(NC(=O)[C@H](N)CCP(=O)(O)O)c1,1
2,N=C(NC(=O)C(NC(=O)c1ccc(Cl)c(Cl)c1)c1ccc(Cl)c(...,1
3,CC(C)(C)OC(=O)NC(C(=O)NC(C(=O)NC(=N)c1ccccc1)c...,1
4,CCOC(=O)NC(C(=O)NC(C(=O)NC(=N)c1ccccc1)c1cccc(...,0


In [8]:
## count active and inactive
data['Activity'].value_counts()

,count
Activity,
1,3399
0,623


In [9]:
## Molecular Descriptors

# descriptor function to include physicochemical and topological descriptors

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski

# Function to compute extended descriptors
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        # Return NaN for invalid SMILES
        return pd.Series([np.nan]*11,
                         index=["MolWt", "LogP", "NumHDonors", "NumHAcceptors",
                                "NumRings", "NumRotatableBonds", "NumHeavyAtoms",
                                "NumHeteroatoms", "NumAromaticRings", "TPSA", "FractionCSP3"])

    # Compute descriptors
    return pd.Series([
        Descriptors.MolWt(mol),                  # Molecular Weight
        Crippen.MolLogP(mol),                    # LogP (hydrophobicity)
        Lipinski.NumHDonors(mol),                # H-bond donors
        Lipinski.NumHAcceptors(mol),             # H-bond acceptors
        Lipinski.RingCount(mol),                 # Number of rings
        Lipinski.NumRotatableBonds(mol),         # Rotatable bonds
        Descriptors.HeavyAtomCount(mol),         # Number of heavy atoms
        Descriptors.NumHeteroatoms(mol),         # Number of heteroatoms
        Lipinski.NumAromaticRings(mol),          # Number of aromatic rings
        Descriptors.TPSA(mol),                   # Topological Polar Surface Area
        Lipinski.FractionCSP3(mol)               # Fraction of sp3 carbons
    ], index=["MolWt", "LogP", "NumHDonors", "NumHAcceptors",
              "NumRings", "NumRotatableBonds", "NumHeavyAtoms",
              "NumHeteroatoms", "NumAromaticRings", "TPSA", "FractionCSP3"])

# Apply descriptor calculation to all SMILES
desc_df = data["Smiles"].apply(compute_descriptors)

In [10]:
desc_df.head()

,MolWt,LogP,NumHDonors,NumHAcceptors,NumRings,NumRotatableBonds,NumHeavyAtoms,NumHeteroatoms,NumAromaticRings,TPSA,FractionCSP3
0,421.537,4.57270,1.0,4.0,4.0,8.0,31.0,5.0,2.0,59.00,0.423077
1,370.430,3.42320,4.0,3.0,1.0,12.0,25.0,7.0,1.0,112.65,0.611111
2,495.193,5.91307,3.0,3.0,3.0,5.0,31.0,9.0,3.0,82.05,0.045455
3,555.462,5.55847,4.0,5.0,3.0,7.0,38.0,10.0,3.0,120.38,0.214286
4,527.408,4.77987,4.0,5.0,3.0,8.0,36.0,10.0,3.0,120.38,0.153846


In [11]:
# Merge descriptors with original dataframe
data = pd.concat([data, desc_df], axis=1)
data.head()

,Smiles,Activity,MolWt,LogP,NumHDonors,NumHAcceptors,NumRings,NumRotatableBonds,NumHeavyAtoms,NumHeteroatoms,NumAromaticRings,TPSA,FractionCSP3
0,CCc1ccc(COc2ccc3c(c2)CCC(CN2CC(C(=O)O)C2)=C3C)...,1,421.537,4.57270,1.0,4.0,4.0,8.0,31.0,5.0,2.0,59.00,0.423077
1,CCCCCCCCc1cccc(NC(=O)[C@H](N)CCP(=O)(O)O)c1,1,370.430,3.42320,4.0,3.0,1.0,12.0,25.0,7.0,1.0,112.65,0.611111
2,N=C(NC(=O)C(NC(=O)c1ccc(Cl)c(Cl)c1)c1ccc(Cl)c(...,1,495.193,5.91307,3.0,3.0,3.0,5.0,31.0,9.0,3.0,82.05,0.045455
3,CC(C)(C)OC(=O)NC(C(=O)NC(C(=O)NC(=N)c1ccccc1)c...,1,555.462,5.55847,4.0,5.0,3.0,7.0,38.0,10.0,3.0,120.38,0.214286
4,CCOC(=O)NC(C(=O)NC(C(=O)NC(=N)c1ccccc1)c1cccc(...,0,527.408,4.77987,4.0,5.0,3.0,8.0,36.0,10.0,3.0,120.38,0.153846


In [12]:
# Drop any rows with invalid SMILES
data.dropna(inplace=True)
data.head()

,Smiles,Activity,MolWt,LogP,NumHDonors,NumHAcceptors,NumRings,NumRotatableBonds,NumHeavyAtoms,NumHeteroatoms,NumAromaticRings,TPSA,FractionCSP3
0,CCc1ccc(COc2ccc3c(c2)CCC(CN2CC(C(=O)O)C2)=C3C)...,1,421.537,4.57270,1.0,4.0,4.0,8.0,31.0,5.0,2.0,59.00,0.423077
1,CCCCCCCCc1cccc(NC(=O)[C@H](N)CCP(=O)(O)O)c1,1,370.430,3.42320,4.0,3.0,1.0,12.0,25.0,7.0,1.0,112.65,0.611111
2,N=C(NC(=O)C(NC(=O)c1ccc(Cl)c(Cl)c1)c1ccc(Cl)c(...,1,495.193,5.91307,3.0,3.0,3.0,5.0,31.0,9.0,3.0,82.05,0.045455
3,CC(C)(C)OC(=O)NC(C(=O)NC(C(=O)NC(=N)c1ccccc1)c...,1,555.462,5.55847,4.0,5.0,3.0,7.0,38.0,10.0,3.0,120.38,0.214286
4,CCOC(=O)NC(C(=O)NC(C(=O)NC(=N)c1ccccc1)c1cccc(...,0,527.408,4.77987,4.0,5.0,3.0,8.0,36.0,10.0,3.0,120.38,0.153846


In [13]:
## Morgan Fingerprint generator
from rdkit.Chem import rdFingerprintGenerator

# Create a 1024-bit Morgan fingerprint generator (ECFP4, radius=2)
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

# Function to generate Morgan fingerprint
def smiles_to_morgan(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(1024)  # Return zeros for invalid SMILES
    return morgan_gen.GetFingerprintAsNumPy(mol)

# Apply fingerprint generation
data['MorganFingerprint'] = data['Smiles'].apply(smiles_to_morgan)

In [14]:
# Convert fingerprints to numpy array for ML
import numpy as np
X = np.vstack(data['MorganFingerprint'].values)
y = data['Activity'].values

print("Morgan fingerprints generated:", X.shape)
print("Activity labels:", y.shape)

Morgan fingerprints generated: (4022, 1024)
Activity labels: (4022,)


In [15]:
# Save fingerprints  ML
fingerprint_df = pd.DataFrame(X)
fingerprint_df['Activity'] = y

In [16]:
fingerprint_df.head()

,0,1,2,3,4,5,6,7,8,9,...,1015,1016,1017,1018,1019,1020,1021,1022,1023,Activity
0,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,1,0,1,0,0,1
1,0,1,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [17]:
## Train/Test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train set:", X_train.shape, "Test set:", X_test.shape)

Train set: (3217, 1024) Test set: (805, 1024)


In [18]:
# Apply SMOTE (for imbalance) (Active > Inactive), use SMOTE on the training set only:

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Check new class distribution
from collections import Counter
print(Counter(y_train_res))

Counter({np.int64(0): 2727, np.int64(1): 2727})


In [19]:
# Save processed data
# Use X_train_res.pkl and y_train_res.pkl for Training only and not for testing
# Use X_test.pkl and y_test.pkl for testing

import pickle

# Save original train/test splits
with open("X_train.pkl", "wb") as f:
    pickle.dump(X_train, f)
with open("X_test.pkl", "wb") as f:
    pickle.dump(X_test, f)
with open("y_train.pkl", "wb") as f:
    pickle.dump(y_train, f)
with open("y_test.pkl", "wb") as f:
    pickle.dump(y_test, f)

# Save SMOTE-resampled training set
with open("X_train_res.pkl", "wb") as f:
    pickle.dump(X_train_res, f)
with open("y_train_res.pkl", "wb") as f:
    pickle.dump(y_train_res, f)

print("Features extracted, SMOTE applied, and data prepared for ML.")

Features extracted, SMOTE applied, and data prepared for ML.
